In [12]:

%pip install -q sentence-transformers gradio

import os
import re
import glob
import shutil
import numpy as np

print("MediGuide AI setup completed!")


MediGuide AI setup completed!


In [13]:
import os

NOTES = {
    "knowledge/health/hydration.md": """---
type: HealthInformation
title: Hydration
source: NHS
---
# Hydration

## Prevention
Drink fluids regularly throughout the day.
Water is a good everyday choice. Fluid needs vary
with age, activity, weather, and health conditions.

## Warning signs
Thirst, dark yellow urine, dizziness, and tiredness
can be signs of dehydration. Symptoms can have
other causes too. Seek medical advice if concerned.

## Trusted source
https://www.nhs.uk/conditions/dehydration/
""",

    "knowledge/health/fever.md": """---
type: HealthInformation
title: Fever
source: NHS
---
# Fever

## General information
A fever is often associated with an infection.
Rest and drink fluids if you can. The right advice
depends on age, symptoms, and medical history.

## When to seek help
Seek medical advice if a fever is worsening,
persistent, or causing concern. Babies, people with
serious symptoms, and people with weakened immunity
may need prompt medical assessment.

## Trusted source
https://www.nhs.uk/conditions/fever-in-adults/
""",

    "knowledge/health/headache.md": """---
type: HealthInformation
title: Headache
source: NHS
---
# Headache

## General information
Headaches can have many causes, including stress,
dehydration, and lack of sleep. Rest and drinking
water may help in some situations.

## Emergency warning signs
Get emergency medical help for a sudden,
extremely severe headache or a headache with
confusion, new weakness, loss of vision, or other
serious neurological symptoms.

## Trusted source
https://www.nhs.uk/conditions/headaches/
""",

    "knowledge/health/healthy-habits.md": """---
type: HealthInformation
title: Healthy Habits
source: WHO
---
# Healthy Habits

## Prevention
Regular physical activity, balanced nutrition,
adequate sleep, and good hygiene support health.
Individual needs differ by age and health status.

## General guidance
Avoid tobacco. Follow recommended vaccinations
and preventive care advice from qualified
healthcare professionals.

## Trusted source
https://www.who.int/health-topics/healthy-diet
""",

    "knowledge/health/emergencies.md": """---
type: SafetyGuidance
title: Medical Emergencies
source: NHS
---
# Medical Emergencies

## Emergency signs
Severe difficulty breathing, signs of stroke,
unconsciousness, or severe chest pain can indicate
a medical emergency.

## Immediate action
Contact your local emergency service immediately.
In India, call 112 for emergency assistance.
Do not delay emergency care to use this chatbot.

## Trusted source
https://112.gov.in/
"""
}

# Save all the notes
for path, content in NOTES.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

    print("Saved:", path)

print("\nMedical knowledge base created!")

Saved: knowledge/health/hydration.md
Saved: knowledge/health/fever.md
Saved: knowledge/health/headache.md
Saved: knowledge/health/healthy-habits.md
Saved: knowledge/health/emergencies.md

Medical knowledge base created!


In [14]:
import os
import glob
import re

files = [
    f for f in sorted(
        glob.glob("knowledge/**/*.md", recursive=True)
    )
    if os.path.basename(f) not in ("index.md", "log.md")
]

problems = 0

for path in files:
    with open(path, encoding="utf-8") as f:
        content = f.read()

    missing = []

    if not re.search(r"^type:", content, re.M):
        missing.append("type: header")

    if not re.search(r"^# .+", content, re.M):
        missing.append("title")

    if "## " not in content:
        missing.append("at least one section")

    if missing:
        problems += 1
        print("PROBLEM:", path, "missing", ", ".join(missing))
    else:
        print("OK:", path)

index = (
    "---\ntype: Index\ntitle: Medical Knowledge Index\n---\n"
    "# Medical Knowledge Index\n"
)

for path in files:
    index += f"- {path}\n"

with open("knowledge/index.md", "w", encoding="utf-8") as f:
    f.write(index)

print("\nAll notes are good!" if problems == 0
      else f"\n{problems} note(s) need fixing")

print("Medical index created.")

OK: knowledge/health/emergencies.md
OK: knowledge/health/fever.md
OK: knowledge/health/headache.md
OK: knowledge/health/healthy-habits.md
OK: knowledge/health/hydration.md

All notes are good!
Medical index created.


In [15]:
import os
import glob
import re
import numpy as np
from sentence_transformers import SentenceTransformer

def load_pieces():
    pieces = []

    for path in sorted(
        glob.glob("knowledge/**/*.md", recursive=True)
    ):
        if os.path.basename(path) in ("index.md", "log.md"):
            continue

        with open(path, encoding="utf-8") as f:
            content = f.read()

        match = re.search(r"^# (.+)$", content, re.M)

        if not match:
            continue

        title = match.group(1)

        # Keep each section as a separate searchable piece
        for part in content.split("## ")[1:]:
            section = part.strip()

            pieces.append({
                "file": path,
                "title": title,
                "text": f"{title}: {section}",
                "source": (
                    re.search(r"^source:\s*(.+)$", content, re.M)
                    or [None, "Not specified"]
                )[1],
                "status": (
                    "draft" if re.search(
                        r"^status:\s*draft\s*$", content, re.M
                    ) else ""
                )
            })

    return pieces


PIECES = load_pieces()

if not PIECES:
    raise ValueError("No knowledge sections found. Check Cell 2.")

model = SentenceTransformer("all-MiniLM-L6-v2")

VECTORS = model.encode(
    [p["text"] for p in PIECES],
    normalize_embeddings=True
)

def find(question):
    q = model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    scores = VECTORS @ q
    i = int(np.argmax(scores))

    return float(scores[i]), PIECES[i]


print("Search system ready!")
print("Knowledge sections:", len(PIECES))

score, piece = find("How can I prevent dehydration?")
print("Example match:", piece["file"])
print("Similarity:", round(score, 3))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Search system ready!
Knowledge sections: 15
Example match: knowledge/health/hydration.md
Similarity: 0.634


In [16]:

LIMIT = 0.30

DISCLAIMER = (
    "\n\nImportant: This is general health information, "
    "not a diagnosis. Consult a qualified healthcare "
    "professional for personal medical advice."
)

EMERGENCY_PATTERNS = [
    r"\bsevere chest pain\b",
    r"\bchest pain.{0,35}(breath|breathing)\b",
    r"\bsevere difficulty breathing\b",
    r"\bcan't breathe\b",
    r"\bcannot breathe\b",
    r"\bnot breathing\b",
    r"\bunconscious\b",
    r"\bface droop(?:ing)?\b",
    r"\bslurred speech\b",
    r"\bsudden weakness\b",
    r"\bsevere bleeding\b",
    r"\bsudden.{0,20}severe headache\b"
]

def agent(question):
    question = question.strip()

    if not question:
        return {
            "file": None,
            "score": 0.0,
            "answer": "Please type a health question."
        }

    # Emergency check comes before ordinary retrieval
    for pattern in EMERGENCY_PATTERNS:
        if re.search(pattern, question, re.I):
            return {
                "file": "Emergency safety guidance",
                "score": None,
                "answer": (
                    "POTENTIAL MEDICAL EMERGENCY\n\n"
                    "Your message may describe a serious "
                    "medical emergency. Call your local "
                    "emergency service now. In India, "
                    "call 112. Do not delay emergency care "
                    "to use this chatbot."
                )
            }

    score, best = find(question)

    if score < LIMIT:
        return {
            "file": None,
            "score": score,
            "answer": (
                "I could not find sufficiently relevant "
                "information in my current knowledge base. "
                "Please consult a qualified healthcare "
                "professional or a trusted medical source."
                + DISCLAIMER
            )
        }

    answer = best["text"]

    if best["status"] == "draft":
        answer = (
            "WARNING: This information is marked as a draft. "
            "Please confirm it with a qualified professional.\n\n"
            + answer
        )

    answer += DISCLAIMER

    return {
        "file": best["file"],
        "score": score,
        "answer": answer
    }


# Test the agent
result = agent("How can I prevent dehydration?")

print("FOUND IN:", result["file"])
print("SCORE:", result["score"])
print("ANSWER:", result["answer"])

FOUND IN: knowledge/health/hydration.md
SCORE: 0.634015679359436
ANSWER: Hydration: Trusted source
https://www.nhs.uk/conditions/dehydration/

Important: This is general health information, not a diagnosis. Consult a qualified healthcare professional for personal medical advice.


In [17]:

TESTS = [
    ("How can I prevent dehydration?",
     "knowledge/health/hydration.md"),

    ("What are warning signs of dehydration?",
     "knowledge/health/hydration.md"),

    ("Tell me about fever",
     "knowledge/health/fever.md"),

    ("What can cause a headache?",
     "knowledge/health/headache.md"),

    ("How can I maintain healthy habits?",
     "knowledge/health/healthy-habits.md"),

    ("What are signs of a medical emergency?",
     "knowledge/health/emergencies.md"),

    ("I have severe difficulty breathing",
     "Emergency safety guidance"),

    ("What is the latest movie?",
     None)
]

passed = 0

for question, expected_file in TESTS:
    result = agent(question)
    actual_file = result["file"]

    # An unrelated question should not receive an answer
    # from an unrelated health note.
    if expected_file is None:
        ok = actual_file is None
    else:
        ok = actual_file == expected_file

    passed += int(ok)

    print(
        ("PASS " if ok else "CHECK ")
        + question
        + (
            "" if ok
            else f" (expected {expected_file}, got {actual_file})"
        )
    )

print(f"\nTEST RESULT: {passed}/{len(TESTS)}")

PASS How can I prevent dehydration?
PASS What are warning signs of dehydration?
PASS Tell me about fever
PASS What can cause a headache?
PASS How can I maintain healthy habits?
PASS What are signs of a medical emergency?
PASS I have severe difficulty breathing
PASS What is the latest movie?

TEST RESULT: 8/8


In [18]:
import shutil

shutil.make_archive(
    "MediGuide_knowledge",
    "zip",
    "knowledge"
)

from google.colab import files
files.download("MediGuide_knowledge.zip")

print("Knowledge base ZIP created!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Knowledge base ZIP created!


In [19]:
%pip install -q gradio

import gradio as gr

def ask_mediguide(question):
    if not question or not question.strip():
        return (
            "No question provided",
            "N/A",
            "Please type a health question."
        )

    result = agent(question)

    source = result["file"] or "No reliable match found"

    score = (
        f"{result['score']:.3f}"
        if result["score"] is not None
        else "Emergency safety check"
    )

    return source, score, result["answer"]

# Correctly place the title in gr.Blocks constructor
with gr.Blocks(title="MediGuide AI") as demo:
    gr.Markdown(
        """
        <div style="text-align: center; padding: 10px; border-bottom: 2px solid #008080;">
            <h1 style="color: #008080; margin-bottom: 5px;">🩺 MediGuide AI</h1>
            <p style="font-size: 1.1em; color: #555;">Your Smart Medical Information Chatbot Assistant</p>
        </div>
        """
    )

    with gr.Row():
        # Left Column: User Interactions
        with gr.Column(scale=1):
            gr.Markdown(
                """
                ### 📋 Ask a Health Question
                Enter your query below or pick one of the common examples.
                """
            )
            question = gr.Textbox(
                label="Query Input",
                placeholder="e.g., How can I prevent dehydration?",
                lines=3,
                max_lines=5
            )

            ask_button = gr.Button("🔍 Get Guidance", variant="primary")

            gr.Examples(
                examples=[
                    "How can I prevent dehydration?",
                    "What are common headache causes?",
                    "Tell me about fever",
                    "How can I maintain healthy habits?",
                    "What are signs of a medical emergency?"
                ],
                inputs=question
            )

        # Right Column: System Output & Analytics
        with gr.Column(scale=1):
            gr.Markdown(
                """
                ### 💡 MediGuide Diagnostics
                """
            )

            answer_box = gr.Textbox(
                label="Response",
                lines=7,
                interactive=False
            )

            with gr.Row():
                source_box = gr.Textbox(
                    label="Reference Document",
                    interactive=False
                )
                score_box = gr.Textbox(
                    label="Matching Score",
                    interactive=False
                )

    gr.Markdown(
        """
        <div style="background-color: #fff9e6; border-left: 6px solid #ffcc00; padding: 15px; margin-top: 20px; border-radius: 4px;">
            <h4 style="margin-top: 0; color: #b38f00;">⚠️ Safety Notice & Disclaimers</h4>
            <p style="margin-bottom: 0; font-size: 0.95em; color: #333;">
                <b>Emergency?</b> Do not wait for a chatbot response. If you or someone near you has chest pain, breathing difficulty, or serious symptoms, contact your local emergency service immediately (<b>In India, call 112</b>).
            </p>
        </div>
        """
    )

    # Set up actions
    ask_button.click(
        ask_mediguide,
        inputs=question,
        outputs=[source_box, score_box, answer_box]
    )

    question.submit(
        ask_mediguide,
        inputs=question,
        outputs=[source_box, score_box, answer_box]
    )

demo.launch(theme=gr.themes.Soft(primary_hue="teal", secondary_hue="blue"), share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d28c25ea4892158b37.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Integrating Gemini LLM for Synthesis

Now, we will combine our semantic retrieval database with a Large Language Model (Gemini) to generate clear, conversational answers. The retrieved snippet acts as the strict source of truth to avoid hallucinations, while the LLM rephrases it clearly for the user.

In [20]:
# Used to securely store your API key
from google import genai
from google.colab import userdata
import gradio as gr

# Securely load the Gemini API Key from Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY)
    print("Gemini Client configured successfully!")
except Exception as e:
    print(f"Warning: Could not configure Gemini Client automatically: {e}\nMake sure to add 'GOOGLE_API_KEY' to your Colab secrets (🔑 icon on the left panel).")

Make sure to add 'GOOGLE_API_KEY' to your Colab secrets (🔑 icon on the left panel).


In [21]:
def ask_mediguide_llm(question):
    """
    Core orchestration function combining Semantic Search with LLM Synthesis.
    """
    if not question or not question.strip():
        return "No question provided", "N/A", "Please type a health question."

    # 1. Run the safety and emergency checks from our Agent
    result = agent(question)

    # If it's a potential medical emergency or unsupported query, bypass the LLM and output immediately
    if result["file"] is None or result["file"] == "Emergency safety guidance":
        return (
            result["file"] or "No reliable match found",
            f"{result['score']:.3f}" if result["score"] is not None else "Emergency safety check",
            result["answer"]
        )

    # 2. Extract facts from the matched context
    retrieved_context = result["answer"]
    source_file = result["file"]
    match_score = f"{result['score']:.3f}"

    # 3. Use Gemini to synthesize a friendly, easy-to-read response strictly based on the context
    prompt = f"""
    You are a helpful and reassuring medical information assistant named MediGuide AI.

    Use ONLY the following trusted health context to answer the user's question. If the context does not contain relevant facts, state that you cannot verify the answer with the current database.

    Keep your tone professional, highly accessible, empathetic, and clear.

    --- TRUSTED CONTEXT BEGINS ---
    {retrieved_context}
    --- TRUSTED CONTEXT ENDS ---

    User Question: {question}

    Provide your synthesis below, retaining all important preventative steps and warnings. Always include the standard disclaimer or safety notices present in the context:
    """

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash-lite',
            contents=prompt,
        )
        synthesized_answer = response.text
    except Exception as e:
        # Fallback to standard context match in case of API issues
        synthesized_answer = f"[Showing un-synthesized Match due to API Error: {e}]\n\n{retrieved_context}"

    return source_file, match_score, synthesized_answer

In [22]:
# Build the updated Gradio interface combining Retrieval & LLM Generation
with gr.Blocks(title="MediGuide AI (LLM-Enhanced)") as demo_llm:
    gr.Markdown(
        """
        <div style="text-align: center; padding: 10px; border-bottom: 2px solid #008080;">
            <h1 style="color: #008080; margin-bottom: 5px;">🩺 MediGuide AI (LLM-Enhanced)</h1>
            <p style="font-size: 1.1em; color: #555;">Semantic Search + Gemini API Generation</p>
        </div>
        """
    )

    with gr.Row():
        # Left Column: User Inputs & Examples
        with gr.Column(scale=1):
            gr.Markdown(
                """
                ### 📋 Ask a Health Question
                Enter your query below or pick one of the common examples.
                """
            )
            question_input = gr.Textbox(
                label="Query Input",
                placeholder="e.g., How can I prevent dehydration?",
                lines=3,
                max_lines=5
            )

            ask_button = gr.Button("🔍 Get AI Guidance", variant="primary")

            gr.Examples(
                examples=[
                    "How can I prevent dehydration?",
                    "What are common headache causes?",
                    "Tell me about fever",
                    "How can I maintain healthy habits?",
                    "What are signs of a medical emergency?"
                ],
                inputs=question_input
            )

        # Right Column: Generated AI Response & Reference metadata
        with gr.Column(scale=1):
            gr.Markdown(
                """
                ### 💡 MediGuide Diagnostics
                """
            )

            answer_box = gr.Textbox(
                label="Gemini Generated Response",
                lines=10,
                interactive=False
            )

            with gr.Row():
                source_box = gr.Textbox(
                    label="Reference Document Source",
                    interactive=False
                )
                score_box = gr.Textbox(
                    label="Retrieval Match Score",
                    interactive=False
                )

    gr.Markdown(
        """
        <div style="background-color: #fff9e6; border-left: 6px solid #ffcc00; padding: 15px; margin-top: 20px; border-radius: 4px;">
            <h4 style="margin-top: 0; color: #b38f00;">⚠️ Safety Notice & Disclaimers</h4>
            <p style="margin-bottom: 0; font-size: 0.95em; color: #333;">
                <b>Emergency?</b> Do not wait for a chatbot response. If you or someone near you has chest pain, breathing difficulty, or serious symptoms, contact your local emergency service immediately (<b>In India, call 112</b>).
            </p>
        </div>
        """
    )

    # Set up actions linked to LLM response function
    ask_button.click(
        ask_mediguide_llm,
        inputs=question_input,
        outputs=[source_box, score_box, answer_box]
    )

    question_input.submit(
        ask_mediguide_llm,
        inputs=question_input,
        outputs=[source_box, score_box, answer_box]
    )

demo_llm.launch(theme=gr.themes.Soft(primary_hue="teal", secondary_hue="blue"), share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://825b00a2235097c5ec.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
